In [6]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
import json
from PIL import Image
from collections import defaultdict

# ===================== 配置 =====================
IMAGE_DIR = "/root/autodl-tmp/figures"  # 【混合图专用路径】
FEATURE_SAVE_DIR = "/root/autodl-tmp/mae_new_npz_features"  # 输出：1股票1NPZ
BREAKPOINT_FILE = os.path.join(FEATURE_SAVE_DIR, "mae_breakpoint.json")
# 🔥 原始CSV数据路径
CSV_PATH = "/root/autodl-tmp/日个股数据2.0.csv"

BATCH_SIZE = 32
IMAGE_SIZE = 224
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_FOLDERS = None
LOCAL_CKPT = "/root/autodl-tmp/mae_model/model.safetensors"
MASK_RATIO = 0.75

# ===========================================================================

os.makedirs(FEATURE_SAVE_DIR, exist_ok=True)
print(f"使用设备: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU型号: NVIDIA GeForce RTX 4090 D")
    print(f"GPU 已启用: torch.cuda.is_available() = {torch.cuda.is_available()}")

# ===================== 加载真实标签映射：股票ID_日期 → 收益率Dretwd =====================
def load_label_mapping(csv_path):
    df = pd.read_csv(csv_path)
    df['Trddt'] = pd.to_datetime(df['Trddt']).dt.strftime('%Y%m%d')
    df['key'] = df['Stkcd'].astype(str) + '_' + df['Trddt']
    return dict(zip(df['key'], df['Dretwd']))

# 全局加载标签映射
LABEL_MAP = load_label_mapping(CSV_PATH)

# ===================== 股票ID + 真实标签解析 =====================
def parse_stock_info(filename):
    base = os.path.splitext(filename)[0]
    stock_id = base.split("_")[0]
    label = LABEL_MAP.get(base, 0.0)
    return stock_id, label

# ===================== 数据集 =====================
class StockImageDataset(Dataset):
    def __init__(self, image_dir, transform=None, max_folders=None):
        self.image_dir = image_dir
        self.transform = transform
        self.valid_files = []

        all_subdirs = []
        for root, dirs, files in os.walk(image_dir):
            if root == image_dir:
                all_subdirs = [os.path.join(root, d) for d in dirs]
                break

        if max_folders is not None:
            all_subdirs = all_subdirs[:max_folders]
            print(f"试运行：仅前 {max_folders} 个文件夹")

        print("正在预检查图片有效性...")
        for d in all_subdirs:
            for root, _, files in os.walk(d):
                for f in files:
                    if f.endswith(".png"):
                        path = os.path.join(root, f)
                        try:
                            with Image.open(path) as img:
                                pass
                            self.valid_files.append(path)
                        except:
                            print(f"🗑️  预检查跳过坏图: {path}")

        print(f"✅ 有效图像总数: {len(self.valid_files)}")

    def __len__(self):
        return len(self.valid_files)

    def __getitem__(self, idx):
        path = self.valid_files[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, os.path.basename(path)

# ===================== MAE 模型 + 指标计算 =====================
class MAEEvaluator(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.mask_ratio = MASK_RATIO

    @torch.no_grad()
    def forward(self, x):
        x = x.to(DEVICE)
        B, C, H, W = x.shape

        x_patch = self.model.patch_embed(x)
        cls_token = self.model.cls_token.expand(B, -1, -1)
        x_patch = torch.cat((cls_token, x_patch), dim=1)
        x_patch = x_patch + self.model.pos_embed[:, :x_patch.size(1)]

        num_patches = x_patch.size(1) - 1
        mask_count = int(num_patches * self.mask_ratio)

        noise = torch.rand(B, num_patches, device=DEVICE)
        mask_idx = torch.argsort(noise, dim=1)[:, :mask_count]

        mask = torch.zeros(B, num_patches, device=DEVICE)
        for i in range(B):
            mask[i, mask_idx[i]] = 1
        mask = mask.bool()

        x_visible = x_patch[:, 1:][~mask].view(B, -1, self.model.embed_dim)
        x_visible = torch.cat([x_patch[:, :1], x_visible], dim=1)

        x_enc = self.model.pos_drop(x_visible)
        for blk in self.model.blocks:
            x_enc = blk(x_enc)
        x_enc = self.model.norm(x_enc)

        feat = x_enc[:, 0]
        target = x_patch[:, 1:][mask].view(B, -1, self.model.embed_dim)
        pred = x_enc[:, 1:]

        mse = F.mse_loss(pred, target[:, :pred.size(1)])
        psnr = 10 * torch.log10(1.0 / (mse + 1e-8))

        mean_pred = pred.mean(dim=-1, keepdim=True)
        mean_tar = target[:, :pred.size(1)].mean(dim=-1, keepdim=True)
        var_pred = pred.var(dim=-1, keepdim=True)
        var_tar = target[:, :pred.size(1)].var(dim=-1, keepdim=True)
        cov = ((pred - mean_pred) * (target[:, :pred.size(1)] - mean_tar)).mean(dim=-1, keepdim=True)
        ssim = (2 * mean_pred * mean_tar + 1e-5) * (2 * cov + 1e-5) / ((mean_pred.pow(2) + mean_tar.pow(2) + 1e-5) * (var_pred + var_tar + 1e-5))
        ssim = ssim.mean()

        return feat, mse, psnr, ssim

class MAEFeatureExtractorWithMetrics:
    def __init__(self, device):
        import timm
        import safetensors.torch

        self.model = timm.create_model(
            "vit_base_patch16_224",
            pretrained=False,
            num_classes=0
        )

        state_dict = safetensors.torch.load_file(LOCAL_CKPT, device="cpu")
        self.model.load_state_dict(state_dict, strict=False)
        self.model = self.model.to(device)
        self.model.eval()

        self.evaluator = MAEEvaluator(self.model).to(device)
        print("✅ MAE 模型加载完成 → 支持 MSE / PSNR / SSIM")

    @torch.no_grad()
    def extract(self, x):
        feat, mse, psnr, ssim = self.evaluator(x)
        return feat.cpu().numpy(), mse.item(), psnr.item(), ssim.item()

# ===================== 断点续跑 =====================
def load_breakpoint():
    if os.path.exists(BREAKPOINT_FILE):
        with open(BREAKPOINT_FILE, 'r', encoding='utf-8') as f:
            return set(json.load(f))
    return set()

def save_breakpoint(files):
    with open(BREAKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(list(files), f)

# ===================== 主程序 =====================
if __name__ == "__main__":
    # 🔥 修复：添加 transforms. 前缀
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.225, 0.225])
    ])

    dataset = StockImageDataset(IMAGE_DIR, transform, max_folders=MAX_FOLDERS)
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4
    )

    extractor = MAEFeatureExtractorWithMetrics(DEVICE)
    processed = load_breakpoint()

    # 评估指标统计
    total_mse = 0.0
    total_psnr = 0.0
    total_ssim = 0.0
    count = 0

    # 按股票ID分组存储
    stock_data = defaultdict(lambda: {"feats": [], "labels": []})

    for imgs, names in tqdm(dataloader, desc="提取MAE特征"):
        # 过滤已处理
        valid_img = []
        valid_name = []
        for img, name in zip(imgs, names):
            if name not in processed:
                valid_img.append(img)
                valid_name.append(name)
        if not valid_img:
            continue

        # 批量推理
        tensor = torch.stack(valid_img).to(DEVICE)
        feats, mse, psnr, ssim = extractor.extract(tensor)

        # 累计指标
        total_mse += mse
        total_psnr += psnr
        total_ssim += ssim
        count += 1

        # 按股票ID聚合数据
        for name, feat in zip(valid_name, feats):
            stock_id, label = parse_stock_info(name)
            stock_data[stock_id]["feats"].append(feat)
            stock_data[stock_id]["labels"].append(label)
            processed.add(name)

        # 断点保存
        if len(processed) % (10 * BATCH_SIZE) == 0:
            save_breakpoint(processed)

    # 按股票保存NPZ
    print("\n💾 开始按股票保存MAE特征...")
    for stock_id, data in tqdm(stock_data.items(), desc="保存股票NPZ"):
        feat_arr = np.array(data["feats"])
        label_arr = np.array(data["labels"])
        save_path = os.path.join(FEATURE_SAVE_DIR, f"stock_{stock_id}_mae.npz")
        np.savez(save_path, feature=feat_arr, label=label_arr)

    # 最终保存断点
    save_breakpoint(processed)

    # 输出结果
    print("\n🎉 MAE混合图特征提取完成！")
    print(f"✅ 处理图片总数：{len(processed)} 个")
    print(f"✅ 聚合股票总数：{len(stock_data)} 只")
    print(f"✅ 保存格式：1只股票 = 1个NPZ")
    print(f"📊 平均 MSE：{total_mse/count:.6f}")
    print(f"📊 平均 PSNR：{total_psnr/count:.4f}")
    print(f"📊 平均 SSIM：{total_ssim/count:.4f}")
    print(f"📁 特征路径：{FEATURE_SAVE_DIR}")

使用设备: cuda
GPU型号: NVIDIA GeForce RTX 4090 D
GPU 已启用: torch.cuda.is_available() = True
正在预检查图片有效性...
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/63/63_20211229.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/100/100_20211202.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/338/338_20211231.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/651/651_20211215.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/800/800_20211203.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/786/786_20211228.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/1979/1979_20211124.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/2422/2422_20220211.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/2352/2352_20211213.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/2601/2601_20211020.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/300033/300033_20211216.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/300059/300059_20211206.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/300124/300124_20211213.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/300408/300408_20211206.png
🗑️  预检查跳过坏图: /root/autodl-tmp/figures/300418

提取MAE特征: 100%|██████████| 9717/9717 [10:37<00:00, 15.24it/s]



💾 开始按股票保存MAE特征...


保存股票NPZ: 100%|██████████| 270/270 [00:01<00:00, 144.56it/s]



🎉 MAE混合图特征提取完成！
✅ 处理图片总数：310929 个
✅ 聚合股票总数：270 只
✅ 保存格式：1只股票 = 1个NPZ
📊 平均 MSE：3.348014
📊 平均 PSNR：-5.2459
📊 平均 SSIM：0.0160
📁 特征路径：/root/autodl-tmp/mae_new_npz_features
